# Corn Leaf Disease Classification: VGG16 + RL-Inspired Loss (Google Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harishbala123/corn-disease-classification-vgg16/blob/main/notebooks/colab_training.ipynb)

Baseline VGG16 vs VGG16 trained with an expected-reward (reinforcement-learning-inspired) loss and a hard-negative supervised-contrastive embedding loss; 4 classes: Blight, Common Rust, Gray Leaf Spot, Healthy.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
The notebook clones the repository and runs the same scripts used on Windows, so the code is identical in both environments.
Colab already ships TensorFlow/Keras, so only the missing packages are installed. This avoids replacing Colab's CUDA-matched TensorFlow build.

In [ ]:
!nvidia-smi -L || echo 'No GPU: select Runtime > Change runtime type > T4 GPU'

In [ ]:
import os
if not os.path.exists('/content/corn-disease-classification-vgg16'):
    !git clone -q https://github.com/Harishbala123/corn-disease-classification-vgg16.git /content/corn-disease-classification-vgg16
%cd /content/corn-disease-classification-vgg16
!git log -1 --oneline

In [ ]:
!pip install -q pyyaml opencv-python-headless
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

## 1. Acquire (PlantVillage + PlantDoc corn images), validate with OpenCV, de-duplicate, split by leaf group

In [ ]:
!python scripts/acquire_data.py

## 2. Train the baseline VGG16 (class-weighted cross-entropy)

In [ ]:
!python scripts/train.py --variant baseline

## 3. Train the custom-loss VGG16 (CE + expected-regret + hard-negative SupCon)

In [ ]:
!python scripts/train.py --variant custom

## 4. Evaluate both on the same test set

In [ ]:
!python scripts/evaluate.py
!cat results/metrics/model_comparison.csv

In [ ]:
!python scripts/predict.py --demo --model custom

In [ ]:
from IPython.display import Image, display
display(Image('results/graphs/confusion_matrix_baseline.png'))
display(Image('results/graphs/confusion_matrix_custom.png'))
display(Image('results/graphs/training_curves.png'))
display(Image('results/graphs/per_class_f1.png'))
display(Image('results/graphs/prediction_demo_custom.png'))

In [ ]:
!pip install -q pytest && python -m pytest -q

## Export artifacts
The zip holds the trained model(s), metrics and graphs. Unzip it into the local project folder (Windows) to use
the Colab-trained model with the local evaluation and demo scripts. Mounting Google Drive is optional and only
needed if you want a persistent copy; no credentials are stored in the notebook.

In [ ]:
!zip -qr /content/corn-disease-classification-vgg16_artifacts.zip models results && ls -lh /content/corn-disease-classification-vgg16_artifacts.zip

In [ ]:
SAVE_TO_DRIVE = False   # set True to copy the zip to Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp /content/corn-disease-classification-vgg16_artifacts.zip /content/drive/MyDrive/
else:
    from google.colab import files
    files.download('/content/corn-disease-classification-vgg16_artifacts.zip')